In [1]:
import pandas as pd

clientes = pd.read_csv("../data/raw/clientes.csv", sep=";", decimal=",")
vendedores = pd.read_csv("../data/raw/vendedores.csv", sep=";", decimal=",")
produtos = pd.read_csv("../data/raw/produtos.csv", sep=";", decimal=",")
pedidos = pd.read_csv("../data/raw/pedidos.csv", sep=";", decimal=",")
itens = pd.read_csv("../data/raw/itens_pedido.csv", sep=";", decimal=",")
titulos = pd.read_csv("../data/raw/titulos_receber.csv", sep=";", decimal=",")
pagamentos = pd.read_csv("../data/raw/pagamentos.csv", sep=";", decimal=",")

## Tarefas da limpeza

### 1. Clientes
1. CNPJ: deixar só os números (tirar pontos, barra e traço). Vem antes dos duplicados, senão parte deles fica escondida.
2. Tirar espaços sobrando no começo e no fim dos textos.
3. Cidade "SP": considerar São Paulo.
4. Cidade e segmento com grafias diferentes: unificar para a grafia que mais aparece na base.
5. UF vazia (25): preencher com "SP".
6. Segmento vazio (14) e vendedor_id vazio (9): manter vazio (nulo).
7. Clientes duplicados (28): juntar os cadastros com o mesmo CNPJ, que fica único no banco. Trocar o `cliente_id` antigo pelo novo em `pedidos` e em `titulos`.

### 2. Pedidos
8. 3 pedidos com ano 2052 (133467, 143683, 156644): corrigir para 2025. Vem antes do custo dos produtos, que usa a data do pedido.

### 3. Produtos
9. 6 produtos sem categoria: descobrir a categoria pela descrição.
10. Custo maior que o preço no cadastro (37, 40, 134, 216): corrigir com o custo da última venda (`itens.custo_unitario`, usando a data de `pedidos`).
11. Produtos 901, 902 e 903 (não existem no cadastro): criar como "produto desconhecido", com categoria "Desconhecido".
12. `ativo`: converter S/N para 1/0 (o banco só aceita 0 e 1).

### 4. Itens do pedido
13. 9 itens com quantidade 100 vezes maior: dividir a quantidade por 100.

### 5. Pagamentos
14. 1.195 datas no formato dd/mm/aaaa: converter cada formato separado e juntar com `fillna`. Vem antes dos duplicados, porque 5 deles só ficam iguais depois da conversão.
15. 45 pagamentos duplicados: remover.

### 6. Títulos a receber
16. Status recalculado a partir da soma dos pagamentos (30 estavam errados). Depende dos pagamentos já sem duplicados.
17. 15 títulos de pedidos cancelados: status `CANCELADO`. Vem depois do recálculo, senão o recálculo apaga essa marcação.

### 7. Carga no banco
18. Carregar na ordem das ligações entre tabelas: vendedores → clientes → produtos → pedidos → itens → títulos → pagamentos.

In [15]:
clientes["cnpj"] = clientes["cnpj"].str.replace(r"\D", "", regex=True)
clientes["cnpj"].str.contains(r"[./-]").sum()

np.int64(0)

np.int64(1228)